# LeetCode SQL → pandas: Data Extraction & Engineering Patterns

Notebook ini mengambil soal-soal **database (SQL) LeetCode** yang bertema *data extraction* dan *data engineering*, lalu menerjemahkannya ke **pandas**. Rentangnya lengkap: **Easy → Medium → Hard**.

Filosofinya sama seperti biasa: **tunjukkan hasilnya dulu, jelaskan mesinnya kemudian**. Setiap soal punya struktur tetap:

1. **Referensi soal** — nomor, judul, dan tingkat kesulitan di LeetCode.
2. **Pola** — nama pattern data engineering-nya (anti-join, gaps & islands, as-of lookup, dst).
3. **Tugas** — parafrase singkat dalam Bahasa Indonesia (bukan salinan soal aslinya).
4. **SQL rujukan** — bentuk query-nya, sebagai jangkar mental.
5. **Kode pandas** — data sintetis kecil + solusi + output.

> **Catatan penting.** Deskripsi soal di sini ditulis ulang, dan semua data contoh dibuat sendiri (bukan test case LeetCode). Untuk soal aslinya, buka tautan `leetcode.com/problems/<slug>`.

**Cara pakai:** setiap cell kode berdiri sendiri — data dibuat ulang di tiap soal. Jadi Anda bisa lompat ke soal mana pun tanpa menjalankan cell sebelumnya (kecuali cell setup di bawah).

## Peta terjemahan: SQL → pandas

Tabel ini adalah rangka mental untuk seluruh notebook. Kalau nanti tersesat di satu soal, kembali ke sini.

| Konstruksi SQL | Idiom pandas |
|---|---|
| `SELECT col_a, col_b` | `df[["col_a", "col_b"]]` |
| `WHERE cond` | `df[cond]` — boolean mask |
| `WHERE col IS NULL` | `df[df["col"].isna()]` |
| `WHERE col IN (...)` | `df[df["col"].isin([...])]` |
| `WHERE col NOT IN (...)` | `df[~df["col"].isin([...])]` |
| `DISTINCT` | `.drop_duplicates()` / `.unique()` / `.nunique()` |
| `ORDER BY a, b DESC` | `.sort_values(["a", "b"], ascending=[True, False])` |
| `LIMIT n` | `.head(n)` |
| `INNER JOIN` | `pd.merge(l, r, on=..., how="inner")` |
| `LEFT JOIN` | `pd.merge(l, r, on=..., how="left")` |
| `CROSS JOIN` | `pd.merge(l, r, how="cross")` |
| `LEFT JOIN ... WHERE r.key IS NULL` (anti-join) | `merge(how="left")` lalu filter `.isna()`, atau `~isin()` |
| `GROUP BY k` | `.groupby("k")` |
| `COUNT(*)` | `.size()` |
| `COUNT(col)` | `.count()` — melewati NaN |
| `COUNT(DISTINCT col)` | `.nunique()` |
| `HAVING cond` | filter **setelah** agregasi |
| `CASE WHEN` | `np.select(...)` / `np.where(...)` |
| `COALESCE(x, y)` | `x.fillna(y)` |
| `ROUND(x, n)` | `.round(n)` |
| `LAG(col) OVER (ORDER BY t)` | `.sort_values("t")["col"].shift(1)` |
| `LEAD(col) OVER (ORDER BY t)` | `.sort_values("t")["col"].shift(-1)` |
| `ROW_NUMBER() OVER (PARTITION BY k)` | `.groupby("k").cumcount()` |
| `RANK() / DENSE_RANK() OVER (PARTITION BY k ORDER BY v)` | `.groupby("k")["v"].rank(method="min" / "dense")` |
| `SUM(v) OVER (PARTITION BY k)` | `.groupby("k")["v"].transform("sum")` |
| `SUM(v) OVER (ORDER BY t)` (running total) | `.sort_values("t")["v"].cumsum()` |
| `SUM(v) OVER (ORDER BY t RANGE 6 PRECEDING)` | `.rolling("7D").sum()` di atas `DatetimeIndex` |
| `PIVOT` / `CASE WHEN` + `GROUP BY` | `.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| `UNION ALL` | `pd.concat([a, b])` |
| `UNION` | `pd.concat([a, b]).drop_duplicates()` |
| `DATE_FORMAT(d, '%Y-%m')` | `d.dt.to_period("M")` atau `d.dt.strftime("%Y-%m")` |
| `DATEDIFF(a, b) = 1` | `(a - b) == pd.Timedelta(days=1)` |

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("pandas", pd.__version__)
print("numpy ", np.__version__)


def show(judul, df):
    """Cetak judul lalu tampilkan DataFrame/Series/skalar."""
    print(f"=== {judul} ===")
    if isinstance(df, (pd.DataFrame, pd.Series)):
        display(df)
    else:
        print(df)
    print()

pandas 3.0.5
numpy  2.5.1


---
# Bagian 1 — Easy

Tingkat ini melatih tiga hal: **filtering**, **join dasar**, dan **agregasi dasar**. Yang membuatnya menarik bukan sintaksnya, tapi perbedaan halus semantik antara SQL dan pandas — terutama soal **NULL**.

### E1 · LC 1757 — Recyclable and Low Fat Products · `Easy`

**Pola:** boolean mask majemuk (multi-condition filtering)

**Tugas.** Tabel `Products` punya kolom flag `low_fats` dan `recyclable` berisi `'Y'`/`'N'`. Ambil `product_id` untuk produk yang **kedua** flag-nya `'Y'`.

```sql
SELECT product_id
FROM Products
WHERE low_fats = 'Y' AND recyclable = 'Y';
```

**Jebakan pandas.** Operator logika pada Series adalah `&`, `|`, `~` — **bukan** `and`, `or`, `not`. Dan setiap kondisi wajib dibungkus tanda kurung karena `&` punya presedensi lebih tinggi daripada `==`.

In [2]:
products = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "low_fats":   ["Y", "Y", "N", "Y", "N"],
    "recyclable": ["N", "Y", "Y", "Y", "N"],
})
show("Products", products)

# SALAH: products[products["low_fats"] == "Y" and products["recyclable"] == "Y"]
#        -> ValueError, karena `and` memaksa Series jadi satu nilai boolean.

mask = (products["low_fats"] == "Y") & (products["recyclable"] == "Y")
hasil = products.loc[mask, ["product_id"]]
show("E1 — hasil", hasil)

assert hasil["product_id"].tolist() == [2, 4]

=== Products ===


,product_id,low_fats,recyclable
0,1,Y,N
1,2,Y,Y
2,3,N,Y
3,4,Y,Y
4,5,N,N



=== E1 — hasil ===


,product_id
1,2
3,4


### E2 · LC 584 — Find Customer Referee · `Easy`

**Pola:** semantik NULL pada perbandingan — jebakan paling mahal saat pindah SQL ↔ pandas

**Tugas.** Tabel `Customer(id, name, referee_id)`. Ambil `name` pelanggan yang **tidak** direferensikan oleh pelanggan dengan `id = 2`. Pelanggan tanpa perekomendasi (`referee_id` NULL) **ikut terhitung**.

```sql
SELECT name
FROM Customer
WHERE referee_id <> 2 OR referee_id IS NULL;
```

**Inti pelajarannya.** Di SQL, `NULL <> 2` bernilai `UNKNOWN`, jadi baris NULL **terbuang** — makanya `OR referee_id IS NULL` wajib ditulis.

Di pandas arahnya justru **terbalik**: `np.nan != 2` bernilai `True`, jadi baris NaN **ikut terbawa** tanpa Anda minta. Kebetulan di soal ini hasilnya benar — tapi itu kebetulan, bukan pemahaman. Perhatikan juga bahwa `<` dan `>` berperilaku berbeda dari `!=` terhadap NaN (lihat E7).

In [3]:
customer = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6],
    "name":       ["Sari", "Bima", "Rani", "Dewi", "Agus", "Tono"],
    "referee_id": [np.nan, np.nan, 2.0, 3.0, 2.0, np.nan],
})
show("Customer", customer)

# Perbandingan NaN di pandas: != menghasilkan True, sedangkan < > == menghasilkan False.
print("np.nan != 2 ->", customer["referee_id"].ne(2).tolist())
print("np.nan <  2 ->", customer["referee_id"].lt(2).tolist())
print()

# Tulis eksplisit agar niatnya terbaca, tidak bergantung pada kebetulan.
mask = (customer["referee_id"] != 2) | (customer["referee_id"].isna())
hasil = customer.loc[mask, ["name"]]
show("E2 — hasil", hasil)

assert set(hasil["name"]) == {"Sari", "Bima", "Dewi", "Tono"}

# Peringatan dtype: dengan Int64 (nullable), NA != 2 menghasilkan <NA>, bukan True,
# dan masking dengan <NA> akan melempar error. Selalu .fillna(False) dulu.
nullable = customer.assign(referee_id=customer["referee_id"].astype("Int64"))
print("Int64 -> NA != 2 :", nullable["referee_id"].ne(2).tolist())

=== Customer ===


,id,name,referee_id
0,1,Sari,NaN
1,2,Bima,NaN
2,3,Rani,2.0
3,4,Dewi,3.0
4,5,Agus,2.0
5,6,Tono,NaN



np.nan != 2 -> [True, True, False, True, False, True]
np.nan <  2 -> [False, False, False, False, False, False]

=== E2 — hasil ===


,name
0,Sari
1,Bima
3,Dewi
5,Tono



Int64 -> NA != 2 : [<NA>, <NA>, False, True, False, <NA>]


### E3 · LC 1683 — Invalid Tweets · `Easy`

**Pola:** string accessor `.str`

**Tugas.** Tabel `Tweets(tweet_id, content)`. Ambil `tweet_id` yang panjang `content`-nya lebih dari 15 karakter.

```sql
SELECT tweet_id
FROM Tweets
WHERE CHAR_LENGTH(content) > 15;
```

**Catatan.** Namespace `.str` adalah pintu masuk ke seluruh operasi string vektor di pandas: `.str.len()`, `.str.contains()`, `.str.startswith()`, `.str.extract()`, `.str.split()`. Semuanya aman terhadap NaN (menghasilkan NaN, bukan error).

In [ ]:
tweets = pd.DataFrame({
    "tweet_id": [1, 2, 3, 4],
    "content": [
        "Halo dunia",
        "Belajar pandas itu menyenangkan sekali",
        "Data engineering",
        "Singkat",
    ],
})
tweets["panjang"] = tweets["content"].str.len()
show("Tweets + panjang", tweets)

hasil = tweets.loc[tweets["content"].str.len() > 15, ["tweet_id"]]
show("E3 — hasil", hasil)

assert hasil["tweet_id"].tolist() == [2, 3]

### E4 · LC 1378 — Replace Employee ID With The Unique Identifier · `Easy`

**Pola:** LEFT JOIN untuk *enrichment* (memperkaya kolom, bukan menyaring baris)

**Tugas.** `Employees(id, name)` dan `EmployeeUNI(id, unique_id)`. Untuk setiap karyawan, tampilkan `unique_id` dan `name`. Karyawan tanpa padanan tetap muncul dengan `unique_id` kosong.

```sql
SELECT eu.unique_id, e.name
FROM Employees e
LEFT JOIN EmployeeUNI eu ON e.id = eu.id;
```

**Pemeriksaan wajib setelah setiap merge.** Bandingkan `len(kiri)` dengan `len(hasil)`. Kalau membengkak, kunci join Anda tidak unik di sisi kanan dan baris terduplikasi diam-diam. Gunakan `validate="1:1"` atau `validate="m:1"` supaya pandas yang berteriak, bukan laporan Anda yang salah tiga minggu kemudian.

In [ ]:
employees = pd.DataFrame({
    "id":   [1, 7, 11, 90, 3],
    "name": ["Alice", "Bob", "Meir", "Winston", "Jonathan"],
})
employee_uni = pd.DataFrame({
    "id":        [3, 11, 90],
    "unique_id": [1, 2, 3],
})

hasil = employees.merge(employee_uni, on="id", how="left", validate="1:1")
hasil = hasil[["unique_id", "name"]]
show("E4 — hasil", hasil)

print("baris kiri:", len(employees), "| baris hasil:", len(hasil))
assert len(hasil) == len(employees)
assert hasil["unique_id"].isna().sum() == 2

### E5 · LC 1581 — Customer Who Visited but Did Not Make Any Transactions · `Easy`

**Pola:** **anti-join** — pola paling sering dipakai di data engineering (rekonsiliasi, deteksi record yatim, audit pipeline)

**Tugas.** `Visits(visit_id, customer_id)` dan `Transactions(transaction_id, visit_id, amount)`. Untuk tiap pelanggan, hitung berapa kali ia berkunjung **tanpa** melakukan transaksi sama sekali.

```sql
SELECT customer_id, COUNT(*) AS count_no_trans
FROM Visits
WHERE visit_id NOT IN (SELECT visit_id FROM Transactions)
GROUP BY customer_id;
```

**Dua cara di pandas.** `~isin()` lebih cepat dibaca; `merge(how="left") + isna()` lebih umum karena bekerja juga untuk kunci majemuk. Keduanya ditunjukkan di bawah.

In [7]:
visits = pd.DataFrame({
    "visit_id":    [1, 2, 4, 5, 5, 6, 7, 8],
    "customer_id": [23, 9, 30, 54, 54, 96, 54, 54],
})
transactions = pd.DataFrame({
    "transaction_id": [2, 3, 9, 12],
    "visit_id":       [5, 5, 5, 1],
    "amount":         [310, 300, 200, 910],
})

# Cara 1 — ~isin()
tanpa_trx = visits[~visits["visit_id"].isin(transactions["visit_id"])]

# Cara 2 — left merge + indicator (berlaku juga untuk kunci majemuk)
gab = visits.merge(transactions[["visit_id"]].drop_duplicates(),
                   on="visit_id", how="left", indicator=True)
tanpa_trx_2 = gab[gab["_merge"] == "left_only"]

hasil = (tanpa_trx.groupby("customer_id", as_index=False)
                  .size()
                  .rename(columns={"size": "count_no_trans"}))
show("E5 — hasil", hasil)

assert tanpa_trx_2["visit_id"].tolist() == tanpa_trx["visit_id"].tolist()
assert hasil.set_index("customer_id")["count_no_trans"].to_dict() == {9: 1, 30: 1, 54: 2, 96: 1}

=== E5 — hasil ===


,customer_id,count_no_trans
0,9,1
1,30,1
2,54,2
3,96,1


### E6 · LC 197 — Rising Temperature · `Easy`

**Pola:** perbandingan antar-baris berurutan (`LAG`) dengan **verifikasi jarak tanggal**

**Tugas.** `Weather(id, recordDate, temperature)`. Ambil `id` hari yang suhunya lebih tinggi daripada **hari sebelumnya** — persisnya tanggal kemarin, bukan sekadar baris sebelumnya.

```sql
SELECT w1.id
FROM Weather w1 JOIN Weather w2
  ON DATEDIFF(w1.recordDate, w2.recordDate) = 1
WHERE w1.temperature > w2.temperature;
```

**Jebakan besarnya.** `shift(1)` mengambil **baris sebelumnya**, dan baris sebelumnya belum tentu **hari sebelumnya**. Kalau ada tanggal yang bolong (dan di data nyata selalu ada), `shift` diam-diam membandingkan lintas lubang. Karena itu dua langkah ini tidak bisa ditawar:

1. `sort_values` dulu — `shift` tidak tahu apa-apa soal urutan.
2. Verifikasi selisihnya benar-benar 1 hari — jangan asumsikan.

In [ ]:
weather = pd.DataFrame({
    "id":          [1, 2, 3, 4, 5],
    "recordDate":  pd.to_datetime(["2015-01-01", "2015-01-02", "2015-01-03",
                                   "2015-01-06", "2015-01-07"]),  # 04-05 bolong
    "temperature": [10, 25, 20, 30, 35],
})

w = weather.sort_values("recordDate").reset_index(drop=True)
w["suhu_kemarin"]    = w["temperature"].shift(1)
w["tanggal_kemarin"] = w["recordDate"].shift(1)
w["selisih_hari"]    = w["recordDate"] - w["tanggal_kemarin"]
show("Weather + kolom lag", w)

naik = (w["selisih_hari"] == pd.Timedelta(days=1)) & (w["temperature"] > w["suhu_kemarin"])
hasil = w.loc[naik, ["id"]]
show("E6 — hasil (dengan verifikasi jarak)", hasil)

# Bandingkan dengan versi naif yang melupakan verifikasi jarak:
naif = w.loc[w["temperature"] > w["suhu_kemarin"], ["id"]]
show("E6 — versi naif (SALAH, id=4 lolos padahal lompat 3 hari)", naif)

assert hasil["id"].tolist() == [2, 5]
assert naif["id"].tolist() == [2, 4, 5]

### E7 · LC 577 — Employee Bonus · `Easy`

**Pola:** LEFT JOIN + filter yang sadar-NULL

**Tugas.** `Employee(empId, name, supervisor, salary)` dan `Bonus(empId, bonus)`. Tampilkan nama dan bonus karyawan yang bonusnya **di bawah 1000 atau tidak punya bonus sama sekali**.

```sql
SELECT e.name, b.bonus
FROM Employee e
LEFT JOIN Bonus b ON e.empId = b.empId
WHERE b.bonus < 1000 OR b.bonus IS NULL;
```

**Kontras dengan E2.** Di E2, `!=` **memasukkan** NaN. Di sini `<` **membuang** NaN (`np.nan < 1000` bernilai `False`). Jadi tidak ada aturan tunggal "pandas menyimpan NaN" — perilakunya tergantung operator. Aturan praktisnya: setiap kali sebuah kolom hasil LEFT JOIN muncul di dalam filter, tanyakan secara sadar apa yang harus terjadi pada baris NaN, lalu tuliskan `.isna()` secara eksplisit.

In [ ]:
employee = pd.DataFrame({
    "empId":      [3, 1, 2, 4],
    "name":       ["Brad", "John", "Dan", "Thomas"],
    "supervisor": [np.nan, 3.0, 3.0, 2.0],
    "salary":     [4000, 1000, 2000, 4000],
})
bonus = pd.DataFrame({
    "empId": [2, 4],
    "bonus": [500, 2000],
})

gab = employee.merge(bonus, on="empId", how="left", validate="1:1")
print("np.nan < 1000 ->", gab["bonus"].lt(1000).tolist(), " (NaN jadi False)")
print()

mask = (gab["bonus"] < 1000) | gab["bonus"].isna()
hasil = gab.loc[mask, ["name", "bonus"]]
show("E7 — hasil", hasil)

assert set(hasil["name"]) == {"Brad", "John", "Dan"}

### E8 · LC 596 — Classes With at Least 5 Students · `Easy`

**Pola:** `GROUP BY` + `HAVING` — yaitu **agregasi lalu filter**

**Tugas.** `Courses(student, class)`. Ambil nama kelas yang punya minimal 5 siswa berbeda.

```sql
SELECT class
FROM Courses
GROUP BY class
HAVING COUNT(DISTINCT student) >= 5;
```

**Urutan yang perlu dihafal.** `WHERE` → filter **sebelum** agregasi. `HAVING` → filter **sesudah** agregasi. Di pandas tidak ada dua kata kunci berbeda: keduanya sama-sama boolean mask, yang membedakan hanya posisinya relatif terhadap `.groupby()`.

In [ ]:
courses = pd.DataFrame({
    "student": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "A"],
    "class":   ["Math"]*6 + ["Biology", "Computer", "Math", "Math"],
})

per_kelas = courses.groupby("class", as_index=False)["student"].nunique()
per_kelas = per_kelas.rename(columns={"student": "n_siswa"})
show("Jumlah siswa unik per kelas", per_kelas)

# HAVING = mask SESUDAH agregasi
hasil = per_kelas.loc[per_kelas["n_siswa"] >= 5, ["class"]]
show("E8 — hasil", hasil)

assert hasil["class"].tolist() == ["Math"]

### E9 · LC 1661 — Average Time of Process per Machine · `Easy`

**Pola:** **long → wide** (pivot) sebelum menghitung selisih

**Tugas.** `Activity(machine_id, process_id, activity_type, timestamp)` dengan `activity_type` berisi `'start'` atau `'end'`. Hitung rata-rata durasi proses per mesin, dibulatkan 3 angka desimal.

```sql
SELECT machine_id, ROUND(AVG(t_end - t_start), 3) AS processing_time
FROM (
  SELECT machine_id, process_id,
         MAX(CASE WHEN activity_type = 'start' THEN timestamp END) AS t_start,
         MAX(CASE WHEN activity_type = 'end'   THEN timestamp END) AS t_end
  FROM Activity
  GROUP BY machine_id, process_id
) s
GROUP BY machine_id;
```

**Ide kuncinya.** `start` dan `end` berada di **baris berbeda**, sementara pengurangan butuh keduanya di **satu baris**. Itulah definisi pivot. Di SQL orang menulisnya sebagai `MAX(CASE WHEN ...)`; di pandas ada `pivot_table` yang menyatakan niat yang sama secara langsung.

In [ ]:
activity = pd.DataFrame({
    "machine_id":    [0, 0, 0, 0, 1, 1, 1, 1],
    "process_id":    [0, 0, 1, 1, 0, 0, 1, 1],
    "activity_type": ["start", "end", "start", "end"] * 2,
    "timestamp":     [0.712, 1.520, 3.140, 4.120, 0.550, 1.550, 0.430, 1.420],
})

lebar = activity.pivot_table(index=["machine_id", "process_id"],
                             columns="activity_type",
                             values="timestamp",
                             aggfunc="max").reset_index()
lebar.columns.name = None
lebar["durasi"] = lebar["end"] - lebar["start"]
show("Setelah pivot (long -> wide)", lebar)

hasil = (lebar.groupby("machine_id", as_index=False)["durasi"]
              .mean()
              .rename(columns={"durasi": "processing_time"}))
hasil["processing_time"] = hasil["processing_time"].round(3)
show("E9 — hasil", hasil)

assert hasil["processing_time"].tolist() == [0.894, 0.995]

---
# Bagian 2 — Medium

Di tingkat ini soalnya berhenti menjadi "terjemahkan satu klausa" dan mulai menjadi "susun beberapa langkah". Tema besarnya: **join non-trivial** (cross join, non-equi join, as-of join), **agregasi berkondisi**, dan **window function**.

### M1 · LC 1280 — Students and Examinations · `Medium`

**Pola:** **cross join** untuk membangun kerangka lengkap, lalu LEFT JOIN untuk mengisinya

**Tugas.** `Students(student_id, student_name)`, `Subjects(subject_name)`, `Examinations(student_id, subject_name)`. Hitung berapa ujian yang diikuti tiap siswa untuk **tiap** mata pelajaran, termasuk kombinasi yang jumlahnya nol.

```sql
SELECT s.student_id, s.student_name, sub.subject_name,
       COUNT(e.subject_name) AS attended_exams
FROM Students s
CROSS JOIN Subjects sub
LEFT JOIN Examinations e
       ON e.student_id = s.student_id AND e.subject_name = sub.subject_name
GROUP BY s.student_id, sub.subject_name
ORDER BY s.student_id, sub.subject_name;
```

**Prinsip yang berlaku umum.** Agregasi **tidak pernah** bisa memunculkan kelompok yang tidak ada barisnya. Kalau laporan Anda harus menampilkan nol secara eksplisit (dan laporan bisnis hampir selalu begitu), kerangka lengkapnya harus **dibangun lebih dulu** lewat cross join, baru diisi. Pola ini muncul lagi di M9 dan H6.

In [4]:
students = pd.DataFrame({
    "student_id":   [1, 2, 13],
    "student_name": ["Alice", "Bob", "John"],
})
subjects = pd.DataFrame({"subject_name": ["Math", "Physics", "Programming"]})
examinations = pd.DataFrame({
    "student_id":   [1, 1, 1, 2, 1, 1, 13, 13, 13, 2],
    "subject_name": ["Math", "Physics", "Programming", "Programming",
                     "Physics", "Math", "Math", "Programming", "Physics", "Math"],
})

# 1) kerangka lengkap: setiap siswa x setiap mapel
kerangka = students.merge(subjects, how="cross")
print("ukuran kerangka:", kerangka.shape, "=", len(students), "x", len(subjects))

# 2) hitung ujian yang benar-benar ada
jumlah = (examinations.groupby(["student_id", "subject_name"], as_index=False)
                      .size()
                      .rename(columns={"size": "attended_exams"}))

# 3) tempel ke kerangka, isi yang kosong dengan 0
hasil = kerangka.merge(jumlah, on=["student_id", "subject_name"], how="left")
hasil["attended_exams"] = hasil["attended_exams"].fillna(0).astype(int)
hasil = hasil.sort_values(["student_id", "subject_name"]).reset_index(drop=True)
show("M1 — hasil", hasil)

assert len(hasil) == 9
assert hasil.loc[(hasil.student_id == 2) & (hasil.subject_name == "Physics"),
                 "attended_exams"].item() == 0

ukuran kerangka: (9, 3) = 3 x 3
=== M1 — hasil ===


,student_id,student_name,subject_name,attended_exams
0,1,Alice,Math,2
1,1,Alice,Physics,2
2,1,Alice,Programming,1
3,2,Bob,Math,1
4,2,Bob,Physics,0
5,2,Bob,Programming,1
6,13,John,Math,1
7,13,John,Physics,1
8,13,John,Programming,1


### M2 · LC 570 — Managers with at Least 5 Direct Reports · `Medium`

**Pola:** self-referencing hierarchy + agregasi + lookup balik

**Tugas.** `Employee(id, name, department, managerId)`. Ambil nama manajer yang punya minimal 5 bawahan langsung.

```sql
SELECT name FROM Employee
WHERE id IN (
  SELECT managerId FROM Employee GROUP BY managerId HAVING COUNT(*) >= 5
);
```

**Catatan.** `value_counts()` sudah otomatis membuang NaN, jadi karyawan tanpa atasan tidak perlu disaring manual. Tapi jangan andalkan kebetulan itu — kalau Anda memakai `groupby("managerId")`, NaN juga terbuang secara default, dan kalau Anda memakai `dropna=False` NaN justru menjadi satu kelompok tersendiri. Sadari mana yang sedang Anda pakai.

In [ ]:
employee = pd.DataFrame({
    "id":         [101, 102, 103, 104, 105, 106, 107],
    "name":       ["John", "Dan", "James", "Amy", "Anne", "Ron", "Sara"],
    "department": ["A", "A", "A", "A", "A", "B", "A"],
    "managerId":  [np.nan, 101, 101, 101, 101, 101, 102],
})

n_bawahan = employee["managerId"].value_counts()
show("Jumlah bawahan langsung", n_bawahan)

manajer_id = n_bawahan[n_bawahan >= 5].index
hasil = employee.loc[employee["id"].isin(manajer_id), ["name"]]
show("M2 — hasil", hasil)

assert hasil["name"].tolist() == ["John"]

### M3 · LC 1934 — Confirmation Rate · `Medium`

**Pola:** rasio berbasis LEFT JOIN — pembilang dari tabel event, penyebut dari tabel dimensi

**Tugas.** `Signups(user_id, time_stamp)` dan `Confirmations(user_id, time_stamp, action)` dengan `action` bernilai `'confirmed'`/`'timeout'`. Hitung rasio konfirmasi tiap pengguna (jumlah `confirmed` dibagi total percobaan), bulatkan 2 desimal. Pengguna yang tidak pernah mencoba mendapat rasio `0.00`.

```sql
SELECT s.user_id,
       ROUND(AVG(IF(c.action = 'confirmed', 1, 0)), 2) AS confirmation_rate
FROM Signups s
LEFT JOIN Confirmations c ON s.user_id = c.user_id
GROUP BY s.user_id;
```

**Kenapa LEFT JOIN-nya penting.** Pengguna tanpa baris konfirmasi tetap harus muncul dengan nilai 0. Setelah LEFT JOIN, mereka menyisakan satu baris dengan `action` bernilai NaN; `(action == "confirmed")` menghasilkan `False`, dan rata-rata dari satu `False` adalah 0.0. Persis yang diminta — tapi periksa sendiri, jangan percaya kebetulan.

In [5]:
signups = pd.DataFrame({
    "user_id":    [3, 7, 2, 6],
    "time_stamp": pd.to_datetime(["2020-03-21 10:16:13", "2020-01-04 13:57:59",
                                  "2020-07-29 23:09:44", "2020-12-09 10:39:37"]),
})
confirmations = pd.DataFrame({
    "user_id":    [3, 3, 7, 7, 7, 2],
    "time_stamp": pd.to_datetime(["2021-01-06 03:30:46", "2021-07-14 14:00:00",
                                  "2021-06-12 11:57:29", "2021-06-13 12:58:28",
                                  "2021-06-14 13:59:27", "2021-01-22 00:00:00"]),
    "action":     ["timeout", "timeout", "confirmed", "confirmed", "confirmed", "timeout"],
})

gab = signups.merge(confirmations, on="user_id", how="left", suffixes=("_signup", "_conf"))
gab["is_confirmed"] = (gab["action"] == "confirmed").astype(int)
show("Setelah LEFT JOIN", gab[["user_id", "action", "is_confirmed"]])

hasil = (gab.groupby("user_id", as_index=False)["is_confirmed"]
            .mean()
            .rename(columns={"is_confirmed": "confirmation_rate"}))
hasil["confirmation_rate"] = hasil["confirmation_rate"].round(2)
show("M3 — hasil", hasil)

assert hasil.set_index("user_id")["confirmation_rate"].to_dict() == {2: 0.0, 3: 0.0, 6: 0.0, 7: 1.0}

=== Setelah LEFT JOIN ===


,user_id,action,is_confirmed
0,3,timeout,0
1,3,timeout,0
2,7,confirmed,1
3,7,confirmed,1
4,7,confirmed,1
5,2,timeout,0
6,6,NaN,0



=== M3 — hasil ===


,user_id,confirmation_rate
0,2,0.0
1,3,0.0
2,6,0.0
3,7,1.0


### M4 · LC 1251 — Average Selling Price · `Medium`

**Pola:** **non-equi join** (join dengan kondisi rentang) + rata-rata tertimbang

**Tugas.** `Prices(product_id, start_date, end_date, price)` dan `UnitsSold(product_id, purchase_date, units)`. Hitung harga jual rata-rata tertimbang tiap produk: total pendapatan dibagi total unit. Produk tanpa penjualan bernilai 0.

```sql
SELECT p.product_id,
       IFNULL(ROUND(SUM(p.price * u.units) / SUM(u.units), 2), 0) AS average_price
FROM Prices p
LEFT JOIN UnitsSold u
       ON p.product_id = u.product_id
      AND u.purchase_date BETWEEN p.start_date AND p.end_date
GROUP BY p.product_id;
```

**Ini pola paling penting di bagian Medium.** pandas **tidak punya** join dengan kondisi rentang. Strateginya selalu dua langkah: join pada kunci kesetaraan (`product_id`), lalu **saring** hasilnya dengan kondisi rentang sebagai boolean mask.

Dua hal yang harus diwaspadai:

- **Ledakan baris.** Hasil sementara berukuran (baris kiri × baris kanan yang cocok kuncinya) sebelum penyaringan. Pada data besar ini bisa meledak — di situlah `merge_asof` atau `IntervalIndex` menjadi perlu (lihat M8).
- **Baris LEFT JOIN yang hilang.** Filter rentang akan membuang baris tanpa padanan, karena `NaT >= start_date` bernilai `False`. Kalau produk tanpa penjualan harus tetap muncul, tambahkan `| purchase_date.isna()` secara eksplisit.

In [ ]:
prices = pd.DataFrame({
    "product_id": [1, 1, 2, 2, 3],
    "start_date": pd.to_datetime(["2019-02-17", "2019-03-01", "2019-02-01", "2019-02-21", "2019-01-01"]),
    "end_date":   pd.to_datetime(["2019-02-28", "2019-03-22", "2019-02-20", "2019-03-31", "2019-12-31"]),
    "price":      [5, 20, 15, 30, 99],
})
units_sold = pd.DataFrame({
    "product_id":    [1, 1, 2, 2],
    "purchase_date": pd.to_datetime(["2019-02-25", "2019-03-01", "2019-02-10", "2019-03-22"]),
    "units":         [100, 15, 200, 30],
})

# Langkah 1 — join pada kunci kesetaraan saja
kasar = prices.merge(units_sold, on="product_id", how="left")
print("baris sebelum filter rentang:", len(kasar))

# Langkah 2 — kondisi rentang sebagai mask; NaT dipertahankan agar produk 3 tidak hilang
dalam_rentang = (kasar["purchase_date"] >= kasar["start_date"]) & \
                (kasar["purchase_date"] <= kasar["end_date"])
cocok = kasar[dalam_rentang | kasar["purchase_date"].isna()].copy()
print("baris sesudah filter rentang:", len(cocok))

cocok["pendapatan"] = cocok["price"] * cocok["units"]
agg = cocok.groupby("product_id", as_index=False).agg(
    total_pendapatan=("pendapatan", "sum"),
    total_unit=("units", "sum"),
)
show("Agregat per produk", agg)

# total_unit == 0 untuk produk tanpa penjualan -> 0/0 = NaN -> isi 0
agg["average_price"] = (agg["total_pendapatan"] / agg["total_unit"]).round(2).fillna(0)
hasil = agg[["product_id", "average_price"]]
show("M4 — hasil", hasil)

assert hasil.set_index("product_id")["average_price"].to_dict() == {1: 6.96, 2: 16.96, 3: 0.0}

### M5 · LC 1193 — Monthly Transactions I · `Medium`

**Pola:** *truncation* tanggal ke periode + **agregasi berkondisi** (conditional aggregation)

**Tugas.** `Transactions(id, country, state, amount, trans_date)` dengan `state` bernilai `'approved'`/`'declined'`. Per bulan per negara, laporkan jumlah transaksi, jumlah yang disetujui, total nominal, dan total nominal yang disetujui.

```sql
SELECT DATE_FORMAT(trans_date, '%Y-%m') AS month, country,
       COUNT(*) AS trans_count,
       SUM(state = 'approved') AS approved_count,
       SUM(amount) AS trans_total_amount,
       SUM(IF(state = 'approved', amount, 0)) AS approved_total_amount
FROM Transactions
GROUP BY month, country;
```

**Teknik intinya.** `SUM(CASE WHEN ... THEN x ELSE 0 END)` di SQL setara dengan **membuat kolom bantu terlebih dahulu**, lalu menjumlahkannya biasa. Menyiapkan kolom indikator sebelum `groupby` hampir selalu lebih terbaca daripada menumpuk `lambda` di dalam `.agg()` — dan jauh lebih cepat, karena `lambda` memaksa pandas keluar dari jalur tervektorisasi.

**Jebakan `groupby`.** `country` bisa NULL. Secara default `groupby` **membuang** baris dengan kunci NaN — laporan Anda akan diam-diam kehilangan baris. Gunakan `dropna=False` kalau NULL adalah kategori yang sah.

In [ ]:
transactions = pd.DataFrame({
    "id":         [121, 122, 123, 124, 125],
    "country":    ["US", "US", "US", "DE", np.nan],
    "state":      ["approved", "declined", "approved", "approved", "approved"],
    "amount":     [1000, 2000, 2000, 2000, 500],
    "trans_date": pd.to_datetime(["2018-12-18", "2018-12-19", "2019-01-01",
                                  "2019-01-07", "2019-01-09"]),
})

t = transactions.copy()
t["month"] = t["trans_date"].dt.strftime("%Y-%m")          # atau .dt.to_period("M")
t["is_approved"]     = (t["state"] == "approved").astype(int)
t["approved_amount"] = t["amount"] * t["is_approved"]      # kolom bantu = CASE WHEN

hasil = (t.groupby(["month", "country"], as_index=False, dropna=False)
          .agg(trans_count=("id", "size"),
               approved_count=("is_approved", "sum"),
               trans_total_amount=("amount", "sum"),
               approved_total_amount=("approved_amount", "sum")))
show("M5 — hasil (dropna=False, baris country NaN ikut)", hasil)

terbuang = t.groupby(["month", "country"], as_index=False).agg(n=("id", "size"))
print("baris dengan dropna default:", terbuang["n"].sum(), "dari", len(t), "transaksi -> 1 hilang")

assert hasil["trans_count"].sum() == 5

### M6 · LC 1174 — Immediate Food Delivery II · `Medium`

**Pola:** ambil **satu baris per grup** (first / top-1-per-group)

**Tugas.** `Delivery(delivery_id, customer_id, order_date, customer_pref_delivery_date)`. Untuk **pesanan pertama** tiap pelanggan, hitung persentase yang bersifat langsung (tanggal pesan = tanggal preferensi), bulatkan 2 desimal.

```sql
SELECT ROUND(AVG(order_date = customer_pref_delivery_date) * 100, 2) AS immediate_percentage
FROM Delivery d
WHERE order_date = (SELECT MIN(order_date) FROM Delivery WHERE customer_id = d.customer_id);
```

**Tiga cara mengambil top-1-per-grup di pandas**, dengan sifat berbeda:

| Cara | Perilaku pada nilai seri (ties) | Catatan |
|---|---|---|
| `loc[groupby.idxmin()]` | ambil satu saja | butuh indeks unik |
| `sort_values().groupby().head(1)` | ambil satu saja | urutan tie-break dapat dikendalikan |
| `rank(method="min") == 1` | **ambil semua** yang seri | ini yang biasanya cocok dengan `RANK()` di SQL |

In [ ]:
delivery = pd.DataFrame({
    "delivery_id":                [1, 2, 3, 4, 5, 6, 7],
    "customer_id":                [1, 2, 1, 3, 3, 2, 4],
    "order_date":                 pd.to_datetime(["2019-08-01", "2019-08-02", "2019-08-11",
                                                  "2019-08-24", "2019-08-21", "2019-08-11",
                                                  "2019-08-09"]),
    "customer_pref_delivery_date": pd.to_datetime(["2019-08-02", "2019-08-02", "2019-08-12",
                                                   "2019-08-24", "2019-08-22", "2019-08-13",
                                                   "2019-08-09"]),
})

pertama = delivery.loc[delivery.groupby("customer_id")["order_date"].idxmin()]
show("Pesanan pertama tiap pelanggan", pertama)

langsung = pertama["order_date"] == pertama["customer_pref_delivery_date"]
hasil = round(langsung.mean() * 100, 2)
show("M6 — immediate_percentage", hasil)

assert hasil == 50.0

### M7 · LC 550 — Game Play Analysis IV · `Medium`

**Pola:** **retensi hari-1** — pola analitik produk yang paling sering ditanyakan

**Tugas.** `Activity(player_id, device_id, event_date, games_played)`. Hitung pecahan pemain yang login lagi **tepat sehari setelah** login pertamanya, bulatkan 2 desimal.

```sql
SELECT ROUND(COUNT(DISTINCT a.player_id) / (SELECT COUNT(DISTINCT player_id) FROM Activity), 2) AS fraction
FROM Activity a
JOIN (SELECT player_id, MIN(event_date) AS first_login FROM Activity GROUP BY player_id) f
  ON a.player_id = f.player_id AND a.event_date = DATE_ADD(f.first_login, INTERVAL 1 DAY);
```

**Cara berpikirnya.** "Apakah pasangan (pemain, tanggal_pertama + 1 hari) ada di tabel aktivitas?" Itu adalah pertanyaan **keanggotaan himpunan**, dan di pandas jawabannya adalah *self-merge*: bangun tabel target, lalu merge balik ke tabel aslinya. Kuncinya majemuk (`player_id` + tanggal), jadi `isin()` tidak cukup — merge-lah alatnya.

In [ ]:
activity = pd.DataFrame({
    "player_id":    [1, 1, 2, 3, 3, 4],
    "device_id":    [2, 2, 3, 1, 4, 1],
    "event_date":   pd.to_datetime(["2016-03-01", "2016-03-02", "2017-06-25",
                                    "2016-03-02", "2018-07-03", "2018-07-03"]),
    "games_played": [5, 6, 1, 0, 5, 5],
})

pertama = (activity.groupby("player_id", as_index=False)["event_date"]
                   .min()
                   .rename(columns={"event_date": "login_pertama"}))
pertama["hari_kedua"] = pertama["login_pertama"] + pd.Timedelta(days=1)
show("Login pertama + target hari kedua", pertama)

kembali = activity.merge(pertama,
                         left_on=["player_id", "event_date"],
                         right_on=["player_id", "hari_kedua"],
                         how="inner")
hasil = round(kembali["player_id"].nunique() / activity["player_id"].nunique(), 2)
show("M7 — fraction (retensi D1)", hasil)

assert hasil == 0.25

### M8 · LC 1164 — Product Price at a Given Date · `Medium`

**Pola:** **as-of lookup** (point-in-time query) — inti dari slowly changing dimension

**Tugas.** `Products(product_id, new_price, change_date)` mencatat riwayat perubahan harga. Ambil harga tiap produk pada tanggal 2019-08-16. Produk yang belum pernah berubah harga sebelum tanggal itu bernilai default 10.

```sql
SELECT product_id, new_price AS price FROM Products
WHERE (product_id, change_date) IN
      (SELECT product_id, MAX(change_date) FROM Products
       WHERE change_date <= '2019-08-16' GROUP BY product_id)
UNION
SELECT DISTINCT product_id, 10 AS price FROM Products
WHERE product_id NOT IN (SELECT product_id FROM Products WHERE change_date <= '2019-08-16');
```

**Ini pertanyaan data engineering yang sesungguhnya.** "Berapa nilainya *pada saat itu*?" muncul di mana-mana: harga historis, kurs, status langganan, snapshot dimensi.

Dua implementasi ditunjukkan. Yang kedua, `pd.merge_asof`, adalah alat khusus untuk pola ini — ia menggabungkan pada **kecocokan terdekat** alih-alih kecocokan persis, sehingga tidak pernah membangun hasil antara yang meledak. Syaratnya: kedua sisi **harus** terurut menurut kunci waktu, kalau tidak pandas akan melempar error (dan itu bagus — kesalahan diam-diam jauh lebih berbahaya).

In [ ]:
products = pd.DataFrame({
    "product_id":  [1, 2, 1, 1, 3],
    "new_price":   [20, 50, 30, 35, 20],
    "change_date": pd.to_datetime(["2019-08-14", "2019-08-14", "2019-08-15",
                                   "2019-08-16", "2019-08-18"]),
})
TANGGAL = pd.Timestamp("2019-08-16")

# --- Cara 1: filter <= tanggal, ambil perubahan terakhir per produk, isi default
riwayat = products[products["change_date"] <= TANGGAL]
idx = riwayat.groupby("product_id")["change_date"].idxmax()
terakhir = riwayat.loc[idx, ["product_id", "new_price"]].rename(columns={"new_price": "price"})

semua_id = products[["product_id"]].drop_duplicates()
hasil = semua_id.merge(terakhir, on="product_id", how="left")
hasil["price"] = hasil["price"].fillna(10).astype(int)
hasil = hasil.sort_values("product_id").reset_index(drop=True)
show("M8 — hasil (cara 1)", hasil)

# --- Cara 2: merge_asof, alat khusus untuk as-of lookup
kiri  = semua_id.assign(asof=TANGGAL).sort_values("asof")
kanan = products.sort_values("change_date")
asof = pd.merge_asof(kiri, kanan,
                     left_on="asof", right_on="change_date",
                     by="product_id", direction="backward")
asof["price"] = asof["new_price"].fillna(10).astype(int)
hasil2 = asof[["product_id", "price"]].sort_values("product_id").reset_index(drop=True)
show("M8 — hasil (cara 2, merge_asof)", hasil2)

pd.testing.assert_frame_equal(hasil, hasil2)
assert hasil.set_index("product_id")["price"].to_dict() == {1: 35, 2: 50, 3: 10}

### M9 · LC 1907 — Count Salary Categories · `Medium`

**Pola:** *binning* + **menjamin semua kategori muncul**, termasuk yang kosong

**Tugas.** `Accounts(account_id, income)`. Hitung jumlah akun pada tiga kategori: `Low Salary` (< 20000), `Average Salary` (20000–50000), `High Salary` (> 50000). Kategori tanpa akun tetap harus muncul dengan nilai 0.

```sql
SELECT 'Low Salary' AS category, COUNT(*) AS accounts_count FROM Accounts WHERE income < 20000
UNION ALL SELECT 'Average Salary', COUNT(*) FROM Accounts WHERE income BETWEEN 20000 AND 50000
UNION ALL SELECT 'High Salary',    COUNT(*) FROM Accounts WHERE income > 50000;
```

**Kenapa `UNION ALL` dipakai di SQL, bukan `GROUP BY`?** Karena `GROUP BY` tidak bisa memunculkan kategori yang nol barisnya — persis masalah yang sama dengan M1.

**Kenapa `np.select`, bukan `pd.cut`?** Batas kategorinya campur: `< 20000` (eksklusif) dan `<= 50000` (inklusif). `pd.cut` memaksa semua batas mengikuti satu arah `right=True/False`, sehingga Anda akan tergoda menulis `19999` — dan diam-diam salah untuk nilai `19999.5`. `np.select` menuliskan kondisi apa adanya. Lalu `.reindex()` memaksa ketiga kategori muncul.

In [ ]:
accounts = pd.DataFrame({
    "account_id": [3, 2, 8, 6],
    "income":     [108939, 12747, 87709, 91796],
})
KATEGORI = ["Low Salary", "Average Salary", "High Salary"]

# np.select: kondisi dievaluasi berurutan, yang pertama cocok menang -> mirip CASE WHEN
accounts["category"] = np.select(
    [accounts["income"] < 20000, accounts["income"] <= 50000],
    ["Low Salary", "Average Salary"],
    default="High Salary",
)
show("Accounts + kategori", accounts)

hasil = (accounts["category"].value_counts()
         .reindex(KATEGORI, fill_value=0)      # <- inilah yang menjamin kategori kosong muncul
         .rename_axis("category")
         .reset_index(name="accounts_count"))
show("M9 — hasil", hasil)

assert hasil.set_index("category")["accounts_count"].to_dict() == \
       {"Low Salary": 1, "Average Salary": 0, "High Salary": 3}

### M10 · LC 1484 — Group Sold Products By The Date · `Medium`

**Pola:** agregasi menjadi **string** (`GROUP_CONCAT` / `STRING_AGG`)

**Tugas.** `Activities(sell_date, product)`. Per tanggal, laporkan jumlah produk unik dan daftar nama produknya, terurut alfabet, dipisah koma.

```sql
SELECT sell_date,
       COUNT(DISTINCT product) AS num_sold,
       GROUP_CONCAT(DISTINCT product ORDER BY product SEPARATOR ',') AS products
FROM Activities
GROUP BY sell_date
ORDER BY sell_date;
```

**Catatan performa.** `lambda` di dalam `.agg()` menonaktifkan jalur tervektorisasi pandas dan berjalan per grup dengan kecepatan Python murni. Untuk agregasi menjadi string memang tidak ada pilihan lain — tapi sadari harganya, dan jangan gunakan `lambda` untuk hal yang punya padanan bawaan (`"sum"`, `"nunique"`, `"max"`).

In [ ]:
activities = pd.DataFrame({
    "sell_date": pd.to_datetime(["2020-05-30", "2020-06-01", "2020-06-02",
                                 "2020-05-30", "2020-06-01", "2020-06-02",
                                 "2020-05-30"]),
    "product":   ["Headphone", "Pencil", "Mask", "Basketball", "Bible", "Mask", "T-Shirt"],
})

hasil = (activities.groupby("sell_date")["product"]
         .agg(num_sold="nunique",
              products=lambda s: ",".join(sorted(s.unique())))
         .reset_index()
         .sort_values("sell_date"))
show("M10 — hasil", hasil)

assert hasil.loc[hasil.sell_date == "2020-05-30", "products"].item() == \
       "Basketball,Headphone,T-Shirt"

### M11 · LC 1204 — Last Person to Fit in the Bus · `Medium`

**Pola:** **running total** (`SUM() OVER (ORDER BY ...)`) + ambang batas

**Tugas.** `Queue(person_id, person_name, weight, turn)`. Bus berkapasitas 1000 kg, orang naik sesuai urutan `turn`. Cari nama orang **terakhir** yang masih bisa naik tanpa melewati kapasitas.

```sql
WITH cte AS (
  SELECT person_name, SUM(weight) OVER (ORDER BY turn) AS total FROM Queue
)
SELECT person_name FROM cte WHERE total <= 1000 ORDER BY total DESC LIMIT 1;
```

**Yang mudah terlewat.** `cumsum()` sama sekali tidak peduli pada urutan logis — ia hanya menjumlahkan **urutan baris fisik** DataFrame. `ORDER BY turn` di SQL adalah bagian dari definisi window; di pandas, `sort_values("turn")` adalah langkah terpisah yang **wajib** ditulis sendiri. Melupakannya menghasilkan angka yang terlihat masuk akal tapi salah — jenis bug paling mahal.

In [ ]:
queue = pd.DataFrame({
    "person_id":   [5, 4, 3, 6, 1, 2],
    "person_name": ["Alice", "Bob", "Alex", "John Cena", "Winston", "Marie"],
    "weight":      [250, 175, 350, 400, 500, 200],
    "turn":        [5, 5, 2, 3, 6, 4],
})
# turn=5 muncul dua kali agar terlihat pentingnya tie-break yang deterministik
queue.loc[queue["person_name"] == "Bob", "turn"] = 1

q = queue.sort_values("turn").reset_index(drop=True)   # WAJIB sebelum cumsum
q["total_kumulatif"] = q["weight"].cumsum()
show("Antrean + berat kumulatif", q)

muat = q[q["total_kumulatif"] <= 1000]
hasil = muat.iloc[-1]["person_name"]
show("M11 — orang terakhir yang muat", hasil)

assert hasil == "John Cena"

---
# Bagian 3 — Hard

Yang membuat soal-soal ini "hard" bukan sintaksnya, melainkan **strukturnya**: Anda harus menyadari dulu bahwa ada pola tersembunyi (gaps & islands, cohort, weighted median) sebelum satu baris kode pun bisa ditulis. Sekali polanya dikenali, kodenya justru pendek.

### H1 · LC 185 — Department Top Three Salaries · `Hard`

**Pola:** **top-N per grup** dengan `DENSE_RANK`

**Tugas.** `Employee(id, name, salary, departmentId)` dan `Department(id, name)`. Untuk tiap departemen, tampilkan karyawan yang gajinya termasuk **tiga nilai gaji tertinggi** di departemen tersebut. Bila beberapa orang bergaji sama, semuanya ikut.

```sql
WITH r AS (
  SELECT e.*, DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rk
  FROM Employee e
)
SELECT d.name AS Department, r.name AS Employee, r.salary AS Salary
FROM r JOIN Department d ON r.departmentId = d.id
WHERE r.rk <= 3;
```

**Pilihan `method` pada `rank()` menentukan jawabannya.** Soal ini meminta tiga **nilai gaji** tertinggi, bukan tiga **orang** teratas — jadi `dense` adalah satu-satunya yang benar.

| `method` | Gaji `[90, 85, 85, 70]` | Arti |
|---|---|---|
| `"dense"` | 1, 2, 2, 3 | tiga *nilai* berbeda teratas → 70 ikut |
| `"min"` (= `RANK`) | 1, 2, 2, 4 | 70 terbuang |
| `"first"` (= `ROW_NUMBER`) | 1, 2, 3, 4 | seri dipecah sembarang |

In [ ]:
employee = pd.DataFrame({
    "id":           [1, 2, 3, 4, 5, 6, 7],
    "name":         ["Joe", "Henry", "Sam", "Max", "Janet", "Randy", "Will"],
    "salary":       [85000, 80000, 60000, 90000, 69000, 85000, 70000],
    "departmentId": [1, 2, 2, 1, 1, 1, 1],
})
department = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})

e = employee.copy()
for m in ["dense", "min", "first"]:
    e[f"rk_{m}"] = e.groupby("departmentId")["salary"].rank(method=m, ascending=False)
show("Perbandingan method rank()", e.sort_values(["departmentId", "salary"], ascending=[True, False]))

e["rk"] = e.groupby("departmentId")["salary"].rank(method="dense", ascending=False)
teratas = e[e["rk"] <= 3]

hasil = (teratas.merge(department.rename(columns={"id": "departmentId", "name": "Department"}),
                       on="departmentId", how="inner")
                .rename(columns={"name": "Employee", "salary": "Salary"})
                [["Department", "Employee", "Salary"]]
                .sort_values(["Department", "Salary"], ascending=[True, False])
                .reset_index(drop=True))
show("H1 — hasil", hasil)

assert set(hasil["Employee"]) == {"Max", "Joe", "Randy", "Will", "Henry", "Sam"}
assert "Janet" not in set(hasil["Employee"])

### H2 · LC 601 — Human Traffic of Stadium · `Hard`

**Pola:** **gaps & islands** — pendeteksian rentetan (run detection)

**Tugas.** `Stadium(id, visit_date, people)`. Tampilkan baris yang menjadi bagian dari rentetan **tiga `id` berurutan atau lebih** yang semuanya punya `people >= 100`.

```sql
WITH ok AS (SELECT *, id - ROW_NUMBER() OVER (ORDER BY id) AS grp FROM Stadium WHERE people >= 100)
SELECT id, visit_date, people FROM ok
WHERE grp IN (SELECT grp FROM ok GROUP BY grp HAVING COUNT(*) >= 3)
ORDER BY visit_date;
```

**Trik yang layak dihafal seumur hidup.** Saring dulu baris yang memenuhi syarat, lalu hitung `id - nomor_baris`. Untuk `id` yang benar-benar berurutan, keduanya naik dengan laju yang sama, sehingga **selisihnya konstan**. Begitu ada `id` yang bolong, selisihnya melompat. Jadi nilai selisih itu **adalah** pengenal kelompok — tanpa loop, tanpa rekursi.

```
id      : 2   3   5   6   7   8
baris ke: 0   1   2   3   4   5
selisih : 2   2   3   3   3   3     <- dua pulau: {2,3} dan {5,6,7,8}
```

Pola ini muncul lagi di H7 dalam bentuk berbasis tanggal.

In [ ]:
stadium = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6, 7, 8],
    "visit_date": pd.to_datetime(["2017-01-01", "2017-01-02", "2017-01-03", "2017-01-04",
                                  "2017-01-05", "2017-01-06", "2017-01-07", "2017-01-09"]),
    "people":     [10, 109, 150, 99, 145, 1455, 199, 188],
})

ramai = stadium[stadium["people"] >= 100].sort_values("id").reset_index(drop=True)
ramai["grup"] = ramai["id"] - np.arange(len(ramai))     # id - ROW_NUMBER()
show("Baris ramai + pengenal pulau", ramai)

ukuran = ramai.groupby("grup")["id"].transform("size")
hasil = (ramai.loc[ukuran >= 3, ["id", "visit_date", "people"]]
              .sort_values("visit_date")
              .reset_index(drop=True))
show("H2 — hasil", hasil)

assert hasil["id"].tolist() == [5, 6, 7, 8]

### H3 · LC 262 — Trips and Users · `Hard`

**Pola:** penyaringan dua sisi terhadap tabel dimensi + tingkat pembatalan harian

**Tugas.** `Trips(id, client_id, driver_id, city_id, status, request_at)` dan `Users(users_id, banned, role)`. Untuk tiap tanggal dalam suatu rentang, hitung tingkat pembatalan — dengan syarat **baik penumpang maupun pengemudinya tidak diblokir**.

```sql
SELECT t.request_at AS Day,
       ROUND(SUM(t.status != 'completed') / COUNT(*), 2) AS 'Cancellation Rate'
FROM Trips t
JOIN Users c ON t.client_id = c.users_id AND c.banned = 'No'
JOIN Users d ON t.driver_id = d.users_id AND d.banned = 'No'
WHERE t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at;
```

**Kenapa ini masuk kategori hard.** Tabel `Users` dipakai **dua kali dengan peran berbeda**. Di SQL itu berarti dua join beralias; di pandas ada versi yang jauh lebih ringkas — bangun himpunan pengguna yang tidak diblokir sekali saja, lalu pakai `isin()` dua kali. Menerjemahkan struktur SQL secara harfiah menghasilkan kode yang berfungsi tapi tidak enak dibaca.

**Perhatikan juga:** urutan operasi penting. Saring dulu (memperkecil data), baru agregasi. Membalik urutannya memberi jawaban yang salah, bukan sekadar lebih lambat.

In [ ]:
trips = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "client_id":  [1, 2, 3, 4, 1, 2, 3, 2, 3, 4],
    "driver_id":  [10, 11, 12, 13, 10, 11, 12, 12, 10, 13],
    "city_id":    [1, 1, 6, 6, 1, 6, 6, 12, 12, 12],
    "status":     ["completed", "cancelled_by_driver", "completed", "cancelled_by_client",
                   "completed", "completed", "completed", "completed",
                   "completed", "cancelled_by_driver"],
    "request_at": pd.to_datetime(["2013-10-01"]*4 + ["2013-10-02"]*3 + ["2013-10-03"]*3),
})
users = pd.DataFrame({
    "users_id": [1, 2, 3, 4, 10, 11, 12, 13],
    "banned":   ["No", "Yes", "No", "No", "No", "No", "No", "No"],
    "role":     ["client"]*4 + ["driver"]*4,
})

tidak_diblokir = users.loc[users["banned"] == "No", "users_id"]

t = trips[trips["client_id"].isin(tidak_diblokir) & trips["driver_id"].isin(tidak_diblokir)]
t = t[t["request_at"].between("2013-10-01", "2013-10-03")].copy()
print("perjalanan sebelum filter:", len(trips), "| sesudah:", len(t))

t["dibatalkan"] = t["status"] != "completed"
hasil = (t.groupby("request_at", as_index=False)["dibatalkan"]
          .mean()
          .rename(columns={"request_at": "Day", "dibatalkan": "Cancellation Rate"}))
hasil["Cancellation Rate"] = hasil["Cancellation Rate"].round(2)
show("H3 — hasil", hasil)

assert hasil["Cancellation Rate"].tolist() == [0.33, 0.0, 0.5]

### H4 · LC 1321 — Restaurant Growth · `Hard`

**Pola:** **rolling window berbasis waktu** (`RANGE BETWEEN INTERVAL 6 DAY PRECEDING`)

**Tugas.** `Customer(customer_id, name, visited_on, amount)`. Untuk tiap tanggal, hitung total belanja tujuh hari terakhir (termasuk hari itu) beserta rata-rata hariannya. Baris hanya muncul mulai hari ke-7.

```sql
SELECT visited_on, amount, ROUND(amount / 7, 2) AS average_amount
FROM (
  SELECT DISTINCT visited_on,
         SUM(amount) OVER (ORDER BY visited_on RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW) AS amount,
         MIN(visited_on) OVER () AS hari_1
  FROM Customer
) x
WHERE visited_on >= hari_1 + INTERVAL 6 DAY;
```

**Perbedaan `ROWS` versus `RANGE` — dan mengapa itu penting.**

- `.rolling(7)` = **7 baris**. Kalau ada tanggal yang tidak punya transaksi, jendelanya diam-diam menjangkau lebih jauh ke belakang dari yang Anda maksud.
- `.rolling("7D")` = **7 hari kalender**, membutuhkan `DatetimeIndex`, dan tetap benar meski ada tanggal bolong.

Untuk metrik berbasis waktu, `"7D"` hampir selalu yang Anda maksudkan. Perhatikan juga langkah pertama: agregasi ke level harian **sebelum** rolling, karena satu tanggal bisa berisi beberapa transaksi.

In [ ]:
tanggal = pd.date_range("2019-01-01", periods=10, freq="D")
customer = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "name":        list("ABCDEFGHIJ"),
    "visited_on":  tanggal,
    "amount":      [100, 110, 120, 130, 110, 140, 150, 80, 110, 100],
})

harian = (customer.groupby("visited_on", as_index=False)["amount"].sum()
                  .sort_values("visited_on")
                  .set_index("visited_on"))

# rolling berbasis waktu: "7D" = 7 hari kalender, bukan 7 baris
harian["total_7d"] = harian["amount"].rolling("7D").sum()
show("Agregat harian + rolling 7 hari", harian)

mulai = harian.index.min() + pd.Timedelta(days=6)      # buang jendela parsial di awal
hasil = harian.loc[harian.index >= mulai, ["total_7d"]].reset_index()
hasil = hasil.rename(columns={"total_7d": "amount"})
hasil["average_amount"] = (hasil["amount"] / 7).round(2)
show("H4 — hasil", hasil)

assert hasil["amount"].iloc[0] == 860
assert len(hasil) == 4

### H5 · LC 1097 — Game Play Analysis V · `Hard`

**Pola:** **analisis kohort** — kelompokkan berdasarkan tanggal instalasi, ukur perilaku relatif terhadapnya

**Tugas.** `Activity(player_id, device_id, event_date, games_played)`. Untuk tiap tanggal instalasi (= tanggal login pertama pemain), laporkan jumlah pemain yang memasang hari itu dan retensi hari-1 mereka, dibulatkan 2 desimal.

```sql
WITH f AS (SELECT player_id, MIN(event_date) AS install_dt FROM Activity GROUP BY player_id)
SELECT f.install_dt AS install_date,
       COUNT(*) AS installs,
       ROUND(COUNT(a.player_id) / COUNT(*), 2) AS Day1_retention
FROM f LEFT JOIN Activity a
  ON a.player_id = f.player_id AND a.event_date = f.install_dt + INTERVAL 1 DAY
GROUP BY f.install_dt;
```

**Bedanya dengan M7.** M7 menghasilkan **satu angka** untuk seluruh populasi. Di sini angka yang sama dipecah **per kohort instalasi** — dan justru inilah yang berguna secara praktis, karena kohort adalah cara untuk melihat apakah retensi membaik atau memburuk dari waktu ke waktu.

`indicator=True` pada `merge` mengembalikan kolom `_merge` bernilai `both` / `left_only` / `right_only`. Ini cara yang bersih untuk menandai "ketemu atau tidak" tanpa mengandalkan kolom mana yang kebetulan berisi NaN.

In [ ]:
activity = pd.DataFrame({
    "player_id":    [1, 1, 2, 3, 3, 4, 4, 5],
    "device_id":    [2, 2, 3, 1, 4, 1, 1, 2],
    "event_date":   pd.to_datetime(["2016-03-01", "2016-03-02", "2017-06-25", "2016-03-01",
                                    "2016-03-02", "2016-03-01", "2016-03-05", "2017-06-25"]),
    "games_played": [5, 6, 1, 0, 5, 5, 2, 3],
})

kohort = (activity.groupby("player_id", as_index=False)["event_date"]
                  .min()
                  .rename(columns={"event_date": "install_date"}))
kohort["hari_kedua"] = kohort["install_date"] + pd.Timedelta(days=1)

ditandai = kohort.merge(
    activity[["player_id", "event_date"]].drop_duplicates(),
    left_on=["player_id", "hari_kedua"],
    right_on=["player_id", "event_date"],
    how="left", indicator=True,
)
ditandai["kembali"] = (ditandai["_merge"] == "both").astype(int)
show("Kohort per pemain", ditandai[["player_id", "install_date", "kembali"]])

hasil = (ditandai.groupby("install_date", as_index=False)
                 .agg(installs=("player_id", "nunique"),
                      Day1_retention=("kembali", "mean")))
hasil["Day1_retention"] = hasil["Day1_retention"].round(2)
show("H5 — hasil", hasil)

assert hasil.set_index("install_date")["installs"].to_dict() == \
       {pd.Timestamp("2016-03-01"): 3, pd.Timestamp("2017-06-25"): 2}
assert hasil["Day1_retention"].tolist() == [0.67, 0.0]

### H6 · LC 1479 — Sales by Day of the Week · `Hard`

**Pola:** pivot dengan **kelengkapan dijamin pada kedua sumbu**

**Tugas.** `Orders(order_id, customer_id, order_date, item_id, quantity)` dan `Items(item_id, item_name, item_category)`. Buat tabel silang: baris = kategori barang, kolom = tujuh hari dalam seminggu, nilai = total kuantitas. Setiap kategori dan setiap hari harus muncul, walaupun nilainya nol.

```sql
SELECT i.item_category AS Category,
       SUM(CASE WHEN DAYNAME(o.order_date) = 'Monday' THEN o.quantity ELSE 0 END) AS Monday,
       ... /* enam kolom serupa */
FROM Items i LEFT JOIN Orders o ON i.item_id = o.item_id
GROUP BY i.item_category;
```

**Inti kesulitannya bukan pivot-nya, melainkan kelengkapan.** `pivot_table` hanya menghasilkan baris dan kolom untuk nilai yang **benar-benar muncul** di data. Kategori tanpa penjualan sama sekali akan hilang; hari yang tidak pernah terjadi transaksi tidak akan punya kolom. Solusinya sama seperti M1 dan M9, tetapi diterapkan pada dua sumbu sekaligus: `reindex(index=..., columns=..., fill_value=0)`.

Perhatikan juga `fill_value=0` di `pivot_table` hanya menambal **sel kosong** di dalam kerangka yang sudah ada — ia tidak bisa memunculkan baris atau kolom yang tidak ada. Kedua mekanisme itu berbeda dan keduanya diperlukan.

In [ ]:
items = pd.DataFrame({
    "item_id":       [1, 2, 3, 4],
    "item_name":     ["LC Alg. Book", "LC DB Book", "LC SmarthPhone", "LC Keychain"],
    "item_category": ["Book", "Book", "Phone", "Merch"],   # Merch tidak pernah terjual
})
orders = pd.DataFrame({
    "order_id":    [1, 2, 3, 4, 5, 6],
    "customer_id": [1, 1, 2, 3, 3, 2],
    "order_date":  pd.to_datetime(["2020-06-01", "2020-06-08", "2020-06-02",
                                   "2020-06-03", "2020-06-04", "2020-06-08"]),
    "item_id":     [1, 2, 1, 3, 3, 1],
})
orders["quantity"] = [10, 1, 5, 2, 3, 7]

HARI = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
KATEGORI = sorted(items["item_category"].unique())

gab = orders.merge(items, on="item_id", how="left")
gab["hari"] = gab["order_date"].dt.day_name()

pivot = gab.pivot_table(index="item_category", columns="hari",
                        values="quantity", aggfunc="sum", fill_value=0)
show("Pivot mentah (Merch hilang, beberapa hari tidak ada kolomnya)", pivot)

hasil = (pivot.reindex(index=KATEGORI, columns=HARI, fill_value=0)
              .fillna(0).astype(int)
              .rename_axis("Category").reset_index())
hasil.columns.name = None
show("H6 — hasil (lengkap 7 hari x semua kategori)", hasil)

assert list(hasil.columns) == ["Category"] + HARI
assert hasil.loc[hasil.Category == "Merch", HARI].to_numpy().sum() == 0
assert hasil.loc[hasil.Category == "Book", "Monday"].item() == 18

### H7 · LC 1225 — Report Contiguous Dates · `Hard`

**Pola:** gaps & islands **berbasis tanggal** dengan penomoran per kategori

**Tugas.** `Failed(fail_date)` dan `Succeeded(success_date)` mencatat status harian sebuah tugas terjadwal. Rangkum menjadi rentang tanggal berurutan yang statusnya sama, terurut menurut tanggal mulai.

```sql
WITH s AS (
  SELECT fail_date AS d, 'failed' AS state FROM Failed
  UNION ALL
  SELECT success_date, 'succeeded' FROM Succeeded
),
g AS (
  SELECT d, state, DATE_SUB(d, INTERVAL ROW_NUMBER() OVER (PARTITION BY state ORDER BY d) DAY) AS grp
  FROM s
)
SELECT state AS period_state, MIN(d) AS start_date, MAX(d) AS end_date
FROM g GROUP BY state, grp ORDER BY start_date;
```

**Variasi dari H2, dengan satu tambahan.** Nomor barisnya di-*partition* per `state`, sehingga rentetan `failed` yang dipotong oleh satu hari `succeeded` benar-benar terpecah menjadi dua pulau.

Cek sendiri kenapa itu bekerja: untuk `failed` pada 01-01 dan 01-03 (dipisah satu hari `succeeded` di 01-02), penomoran per-state memberi `0` dan `1`, sehingga kunci grupnya menjadi `01-01` dan `01-02` — berbeda, jadi terpisah. Sementara `failed` pada 01-01 dan 01-02 menghasilkan kunci `01-01` untuk keduanya — sama, jadi tergabung. Persis yang diinginkan.

In [ ]:
failed = pd.DataFrame({"fail_date": pd.to_datetime(
    ["2018-12-28", "2018-12-29", "2019-01-04", "2019-01-05"])})
succeeded = pd.DataFrame({"success_date": pd.to_datetime(
    ["2018-12-30", "2018-12-31", "2019-01-01", "2019-01-02",
     "2019-01-03", "2019-01-06"])})

f = failed.rename(columns={"fail_date": "d"}).assign(period_state="failed")
s = succeeded.rename(columns={"success_date": "d"}).assign(period_state="succeeded")

semua = pd.concat([f, s], ignore_index=True)                        # UNION ALL
semua = semua[semua["d"].between("2019-01-01", "2019-12-31")]       # hanya tahun 2019
semua = semua.sort_values("d").reset_index(drop=True)

# ROW_NUMBER() OVER (PARTITION BY state ORDER BY d)
semua["rn"] = semua.groupby("period_state").cumcount()
semua["grup"] = semua["d"] - pd.to_timedelta(semua["rn"], unit="D")
show("Tanggal + pengenal pulau per state", semua)

hasil = (semua.groupby(["period_state", "grup"], as_index=False)
              .agg(start_date=("d", "min"), end_date=("d", "max"))
              .sort_values("start_date")
              .drop(columns="grup")
              .reset_index(drop=True))
show("H7 — hasil", hasil)

assert hasil["period_state"].tolist() == ["succeeded", "failed", "succeeded"]
assert hasil["start_date"].astype(str).tolist() == ["2019-01-01", "2019-01-04", "2019-01-06"]

### H8 · LC 571 — Find Median Given Frequency of Numbers · `Hard`

**Pola:** **median tertimbang** melalui cumulative sum — tanpa memekarkan data

**Tugas.** `Numbers(num, frequency)` menyimpan data dalam bentuk terkompresi: tiap nilai beserta berapa kali ia muncul. Hitung median dari data yang direpresentasikannya.

```sql
SELECT AVG(num) AS median FROM (
  SELECT num,
         SUM(frequency) OVER (ORDER BY num) AS c_asc,
         SUM(frequency) OVER (ORDER BY num DESC) AS c_desc,
         SUM(frequency) OVER () AS total
  FROM Numbers
) t
WHERE c_asc >= total / 2 AND c_desc >= total / 2;
```

**Kenapa ini tidak boleh diselesaikan dengan `repeat()`.** Menggembungkan data lalu memanggil `.median()` memang benar secara matematis, tetapi memakan memori sebesar `sum(frequency)` — dan di data nyata, tabel frekuensi justru dipakai **karena** angka itu besar. Solusi cumsum berjalan dalam memori sebesar jumlah nilai unik saja.

**Cara berpikirnya.** Median adalah nilai pada posisi ke-`(n+1)//2` dan `(n+2)//2` (indeks mulai dari 1), lalu dirata-ratakan — rumus tunggal yang menangani `n` ganjil maupun genap. `cumsum()` memberi tahu posisi terakhir yang ditempati tiap nilai, jadi nilai pada posisi ke-`p` adalah nilai pertama yang `cumsum`-nya sudah mencapai `p`.

In [ ]:
numbers = pd.DataFrame({
    "num":       [0, 1, 2, 3],
    "frequency": [7, 1, 3, 1],
})

d = numbers.sort_values("num").reset_index(drop=True)
d["kumulatif"] = d["frequency"].cumsum()
total = int(d["frequency"].sum())
show("Numbers + posisi kumulatif", d)

p1, p2 = (total + 1) // 2, (total + 2) // 2      # menangani ganjil & genap sekaligus
print(f"total n = {total} -> posisi median: {p1} dan {p2}")

def nilai_di_posisi(p):
    """Nilai pertama yang cumsum-nya sudah mencapai posisi p (1-indexed)."""
    return d.loc[d["kumulatif"] >= p, "num"].iloc[0]

median = round((nilai_di_posisi(p1) + nilai_di_posisi(p2)) / 2, 1)
show("H8 — median", median)

# verifikasi dengan cara boros (hanya aman karena datanya kecil)
mekar = np.repeat(d["num"].to_numpy(), d["frequency"].to_numpy())
print("verifikasi via np.repeat:", np.median(mekar))
assert median == np.median(mekar)

---
# Ringkasan jebakan

Delapan hal yang paling sering membuat terjemahan SQL → pandas menghasilkan angka yang **terlihat benar tapi salah**. Semuanya sudah muncul di atas; ini rekapnya.

**1. NULL berperilaku tidak konsisten antar-operator.**
Di SQL, `NULL` apa pun operatornya menghasilkan `UNKNOWN` dan barisnya terbuang. Di pandas, `NaN != x` menghasilkan `True` (baris **ikut**), sedangkan `NaN < x` menghasilkan `False` (baris **terbuang**). Selalu tulis `.isna()` secara eksplisit. → E2, E7

**2. Dtype nullable mengubah aturan lagi.**
Dengan `Int64`/`boolean`, perbandingan terhadap `pd.NA` menghasilkan `NA`, dan masking dengan `NA` melempar error. Gunakan `.fillna(False)` sebelum masking. → E2

**3. `shift()` memakai baris sebelumnya, bukan periode sebelumnya.**
Wajib `sort_values()` dulu, dan wajib verifikasi jarak waktunya benar-benar sesuai. → E6

**4. `groupby` diam-diam membuang kunci NaN.**
Default-nya `dropna=True`. Baris hilang tanpa peringatan apa pun. → M5

**5. Agregasi tidak bisa memunculkan grup kosong.**
Kalau laporan harus menampilkan nol secara eksplisit, bangun kerangka lengkap lebih dulu (cross join), lalu `reindex(fill_value=0)`. → M1, M9, H6

**6. pandas tidak punya join non-equi.**
Join pada kunci kesetaraan dulu, lalu saring dengan mask. Waspadai ledakan baris di tengah jalan, dan waspadai baris LEFT JOIN yang ikut terbuang oleh filter. → M4

**7. `merge` bisa menggandakan baris tanpa suara.**
Selalu bandingkan `len()` sebelum dan sesudah, atau pakai `validate="1:1"` / `"m:1"`. → E4

**8. `rolling(7)` ≠ `rolling("7D")`.**
Yang pertama menghitung baris (`ROWS`), yang kedua menghitung waktu (`RANGE`). Untuk metrik harian, hampir selalu yang kedua yang Anda maksud. → H4

# Pola yang tercakup

| Pola | Soal |
|---|---|
| Boolean mask majemuk | E1 |
| Semantik NULL | E2, E7 |
| String accessor `.str` | E3 |
| LEFT JOIN untuk enrichment | E4 |
| Anti-join | E5 |
| Perbandingan antar-baris (`LAG`) | E6, M11 |
| `GROUP BY` + `HAVING` | E8, M2 |
| Pivot long → wide | E9, H6 |
| Cross join untuk kerangka lengkap | M1, M9, H6 |
| Rasio berbasis LEFT JOIN | M3 |
| Non-equi join (rentang) | M4 |
| Agregasi berkondisi | M5 |
| Top-1 / Top-N per grup | M6, H1 |
| Retensi & analisis kohort | M7, H5 |
| As-of lookup / `merge_asof` | M8 |
| Agregasi string (`GROUP_CONCAT`) | M10 |
| Running total (`cumsum`) | M11, H8 |
| Filter dua sisi ke tabel dimensi | H3 |
| Rolling window berbasis waktu | H4 |
| Gaps & islands | H2, H7 |
| Median tertimbang | H8 |

**Yang belum tercakup dan layak jadi lanjutan:** recursive CTE (hierarki tak terbatas kedalamannya), self-join segitiga (LC 180 Consecutive Numbers), `PERCENTILE_CONT`, dan operasi himpunan (`INTERSECT`/`EXCEPT`) — semuanya punya idiom pandas tersendiri.